In [339]:
import numpy as np
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/ace-aiml-recruitmen-ts-26-1/train.csv")
test = pd.read_csv("/kaggle/input/competitions/ace-aiml-recruitmen-ts-26-1/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/ace-aiml-recruitmen-ts-26-1/sample_submission.csv")

print(train.shape)
print(test.shape)
print(sample.shape)

(2344, 82)
(586, 81)
(586, 2)


In [340]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_log_error

from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

In [341]:
# train = train.drop(
#     train[
#         (train["Gr Liv Area"] > 4000) &
#         (train["SalePrice"] < 300000)
#     ].index
# )

In [342]:
X = train.drop(columns=["SalePrice"])
y = train["SalePrice"]

print(X.shape)
print(y.shape)

(2344, 81)
(2344,)


In [343]:
num_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns

cat_cols = X.select_dtypes(
    include=["object"]
).columns

In [344]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

cat_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Missing"
        )
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [345]:
y_log = np.log1p(y)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y_log,
    test_size=0.2,
    random_state=42
)

In [346]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  Index(['Order', 'PID', 'MS SubClass', 'Lot Frontage', 'Lot Area',
       'Overall Qual', 'Overall Cond', 'Year Built', 'Year Remod/Add',
       'Mas Vnr Area', 'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF',
       'Total Bsmt SF', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF',...
       'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin Type 2', 'Heating', 'Heating QC',
       'Central Air', 'Electrical', 'Kitchen Qual', 'Functional',
       'Fireplace Qu', 'Garage Type', 'Garage Finish', 'Garage Qual',
       'Garage Cond', 'Paved Drive', 'Pool QC', 'Fence', 'Misc Feature',
       'Sale Type', 'Sale Condition'],
      dtype='object'))])),
                ('regressor', LinearRegression())])

In [347]:
pred_log = model.predict(X_valid)

pred = np.expm1(pred_log)
y_valid_original = np.expm1(y_valid)

pred = np.maximum(pred, 0)

msle = mean_squared_log_error(
    y_valid_original,
    pred
)

print("MSLE:", msle)

MSLE: 0.0442702317837075


In [348]:
# from sklearn.ensemble import RandomForestRegressor

# # Same split using original y
# X_train, X_valid, y_train, y_valid = train_test_split(
#     X,
#     y,
#     test_size=0.2,
#     random_state=42
# )

# rf_model = Pipeline([
#     ("preprocessor", preprocessor),
#     ("regressor", RandomForestRegressor(
#         n_estimators=200,
#         random_state=42,
#         n_jobs=-1
#     ))
# ])

# rf_model.fit(
#     X_train,
#     np.log1p(y_train)
# )

In [349]:
# rf_pred_log = rf_model.predict(X_valid)

# rf_pred = np.expm1(rf_pred_log)
# rf_pred = np.maximum(rf_pred, 0)

# rf_msle = mean_squared_log_error(
#     y_valid,
#     rf_pred
# )

# print("Random Forest MSLE:", rf_msle)

In [350]:
# from sklearn.ensemble import GradientBoostingRegressor

# gb_model = Pipeline([
#     ("preprocessor", preprocessor),
#     ("regressor", GradientBoostingRegressor(
#         n_estimators=200,
#         learning_rate=0.05,
#         max_depth=3,
#         random_state=42
#     ))
# ])

# gb_model.fit(
#     X_train,
#     np.log1p(y_train)
# )

In [351]:
# gb_pred_log = gb_model.predict(X_valid)

# gb_pred = np.expm1(gb_pred_log)
# gb_pred = np.maximum(gb_pred, 0)

# gb_msle = mean_squared_log_error(
#     y_valid,
#     gb_pred
# )

# print("Gradient Boosting MSLE:", gb_msle)

In [352]:
# from sklearn.model_selection import KFold
# from sklearn.metrics import mean_squared_log_error

# kf = KFold(
#     n_splits=5,
#     shuffle=True,
#     random_state=42
# )

In [353]:
# gb_scores = []

# for train_idx, valid_idx in kf.split(X):
    
#     X_train_cv = X.iloc[train_idx]
#     X_valid_cv = X.iloc[valid_idx]

#     y_train_cv = y.iloc[train_idx]
#     y_valid_cv = y.iloc[valid_idx]

#     gb_model = Pipeline([
#         ("preprocessor", preprocessor),
#         ("regressor", GradientBoostingRegressor(
#             n_estimators=200,
#             learning_rate=0.05,
#             max_depth=3,
#             random_state=42
#         ))
#     ])

#     gb_model.fit(
#         X_train_cv,
#         np.log1p(y_train_cv)
#     )

#     pred_log = gb_model.predict(X_valid_cv)
#     pred = np.expm1(pred_log)

#     pred = np.maximum(pred, 0)

#     score = mean_squared_log_error(
#         y_valid_cv,
#         pred
#     )

#     gb_scores.append(score)

# print("Gradient Boosting fold scores:")
# print(gb_scores)

# print("Average GB MSLE:", np.mean(gb_scores))

In [354]:
# from sklearn.model_selection import GridSearchCV
# from sklearn.ensemble import GradientBoostingRegressor

# gb_tune_model = Pipeline([
#     ("preprocessor", preprocessor),
#     ("regressor", GradientBoostingRegressor(random_state=42))
# ])

# param_grid = {
#     "regressor__n_estimators": [200, 300],
#     "regressor__learning_rate": [0.03, 0.05],
#     "regressor__max_depth": [2, 3]
# }

# grid_search = GridSearchCV(
#     gb_tune_model,
#     param_grid,
#     cv=5,
#     scoring="neg_mean_squared_error",
#     n_jobs=-1
# )

# grid_search.fit(
#     X,
#     np.log1p(y)
# )

# print("Best parameters:")
# print(grid_search.best_params_)

# print("Best CV score:")
# print(grid_search.best_score_)

In [355]:
# from sklearn.model_selection import KFold
# from sklearn.metrics import mean_squared_log_error

# kf = KFold(
#     n_splits=5,
#     shuffle=True,
#     random_state=42
# )

# tuned_gb_scores = []

# for train_idx, valid_idx in kf.split(X):

#     X_train_cv = X.iloc[train_idx]
#     X_valid_cv = X.iloc[valid_idx]

#     y_train_cv = y.iloc[train_idx]
#     y_valid_cv = y.iloc[valid_idx]

#     tuned_gb_model = Pipeline([
#         ("preprocessor", preprocessor),
#         ("regressor", GradientBoostingRegressor(
#             n_estimators=300,
#             learning_rate=0.05,
#             max_depth=3,
#             random_state=42
#         ))
#     ])

#     tuned_gb_model.fit(
#         X_train_cv,
#         np.log1p(y_train_cv)
#     )

#     pred_log = tuned_gb_model.predict(X_valid_cv)

#     pred = np.expm1(pred_log)
#     pred = np.maximum(pred, 0)

#     score = mean_squared_log_error(
#         y_valid_cv,
#         pred
#     )

#     tuned_gb_scores.append(score)

# print("Tuned Gradient Boosting fold scores:")
# print(tuned_gb_scores)

# print("Average Tuned GB MSLE:", np.mean(tuned_gb_scores))

In [356]:
# final_model = Pipeline([
#     ("preprocessor", preprocessor),
#     ("regressor", GradientBoostingRegressor(
#         n_estimators=300,
#         learning_rate=0.05,
#         max_depth=3,
#         random_state=42
#     ))
# ])

# final_model.fit(
#     X,
#     np.log1p(y)
# )

In [357]:
# test_pred_log = final_model.predict(test)

# test_predictions = np.expm1(test_pred_log)

# test_predictions = np.maximum(test_predictions, 0)

In [358]:
# submission = sample.copy()

# submission["SalePrice"] = test_predictions

# submission.to_csv(
#     "/kaggle/working/submission.csv",
#     index=False
# )

# submission.head()

In [359]:
# print(submission.shape)
# print(submission.isnull().sum())

In [360]:
# import numpy as np

# from sklearn.preprocessing import StandardScaler, OneHotEncoder
# from sklearn.impute import SimpleImputer
# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import mean_squared_log_error

# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Dense, Dropout, Input
# from tensorflow.keras.callbacks import EarlyStopping
# from tensorflow.keras.regularizers import l2
# from tensorflow.keras.optimizers import Adam


# # ---------------------------------------------------------
# # NN-specific preprocessing
# # ---------------------------------------------------------

# num_cols = X.select_dtypes(include=["int64", "float64"]).columns
# cat_cols = X.select_dtypes(include=["object"]).columns

# nn_num_pipeline = Pipeline([
#     ("imputer", SimpleImputer(strategy="median")),
#     ("scaler", StandardScaler())
# ])

# nn_cat_pipeline = Pipeline([
#     (
#         "imputer",
#         SimpleImputer(
#             strategy="constant",
#             fill_value="Missing"
#         )
#     ),
#     (
#         "encoder",
#         OneHotEncoder(
#             handle_unknown="ignore",
#             sparse_output=False
#         )
#     )
# ])

# nn_preprocessor = ColumnTransformer([
#     ("num", nn_num_pipeline, num_cols),
#     ("cat", nn_cat_pipeline, cat_cols)
# ])


# # ---------------------------------------------------------
# # 5-fold neural network CV
# # ---------------------------------------------------------

# nn_scores = []
# nn_oof = np.zeros(len(X))

# for fold, (train_idx, valid_idx) in enumerate(kf.split(X), start=1):

#     print(f"\nFold {fold}")

#     X_train_nn = X.iloc[train_idx]
#     X_valid_nn = X.iloc[valid_idx]

#     y_train_nn = y.iloc[train_idx]
#     y_valid_nn = y.iloc[valid_idx]

#     X_train_processed = nn_preprocessor.fit_transform(X_train_nn)
#     X_valid_processed = nn_preprocessor.transform(X_valid_nn)

#     input_size = X_train_processed.shape[1]

#     nn_model = Sequential([
#         Input(shape=(input_size,)),

#         Dense(
#             256,
#             activation="relu",
#             kernel_regularizer=l2(1e-5)
#         ),
#         Dropout(0.20),

#         Dense(
#             128,
#             activation="relu",
#             kernel_regularizer=l2(1e-5)
#         ),
#         Dropout(0.15),

#         Dense(
#             64,
#             activation="relu",
#             kernel_regularizer=l2(1e-5)
#         ),

#         Dense(1)
#     ])

#     nn_model.compile(
#         optimizer=Adam(learning_rate=0.001),
#         loss="mse"
#     )

#     early_stop = EarlyStopping(
#         monitor="val_loss",
#         patience=20,
#         restore_best_weights=True
#     )

#     nn_model.fit(
#         X_train_processed,
#         np.log1p(y_train_nn),
#         validation_data=(
#             X_valid_processed,
#             np.log1p(y_valid_nn)
#         ),
#         epochs=300,
#         batch_size=32,
#         callbacks=[early_stop],
#         verbose=0
#     )

#     pred_log = nn_model.predict(
#         X_valid_processed,
#         verbose=0
#     ).flatten()

#     # Safety clipping before expm1
#     pred_log = np.clip(pred_log, 0, 20)

#     pred = np.expm1(pred_log)
#     pred = np.maximum(pred, 0)

#     nn_oof[valid_idx] = pred

#     score = mean_squared_log_error(
#         y_valid_nn,
#         pred
#     )

#     nn_scores.append(score)

#     print("NN MSLE:", score)


# print("\nNeural Network fold scores:")
# print(nn_scores)

# print("Average Neural Network MSLE:", np.mean(nn_scores))
# print("NN MSLE std:", np.std(nn_scores))

In [361]:
# # ==========================================
# # Feature Engineering Test
# # ==========================================

# def add_features(df):
#     df = df.copy()

#     df["TotalSF"] = (
#         df["Total Bsmt SF"].fillna(0)
#         + df["1st Flr SF"].fillna(0)
#         + df["2nd Flr SF"].fillna(0)
#     )

#     df["TotalBathrooms"] = (
#         df["Full Bath"].fillna(0)
#         + 0.5 * df["Half Bath"].fillna(0)
#         + df["Bsmt Full Bath"].fillna(0)
#         + 0.5 * df["Bsmt Half Bath"].fillna(0)
#     )

#     df["HouseAge"] = (
#         df["Yr Sold"] - df["Year Built"]
#     )

#     df["RemodAge"] = (
#         df["Yr Sold"] - df["Year Remod/Add"]
#     )

#     df["TotalPorchSF"] = (
#         df["Open Porch SF"].fillna(0)
#         + df["Enclosed Porch"].fillna(0)
#         + df["3Ssn Porch"].fillna(0)
#         + df["Screen Porch"].fillna(0)
#         + df["Wood Deck SF"].fillna(0)
#     )

#     df["HasGarage"] = (
#         df["Garage Area"].fillna(0) > 0
#     ).astype(int)

#     df["HasBasement"] = (
#         df["Total Bsmt SF"].fillna(0) > 0
#     ).astype(int)

#     df["HasFireplace"] = (
#         df["Fireplaces"].fillna(0) > 0
#     ).astype(int)

#     return df


# # Apply feature engineering
# X_fe = add_features(X)

# print("Original X shape:", X.shape)
# print("Feature-engineered X shape:", X_fe.shape)


# # ==========================================
# # Rebuild preprocessing
# # ==========================================

# num_cols_fe = X_fe.select_dtypes(
#     include=["int64", "float64"]
# ).columns

# cat_cols_fe = X_fe.select_dtypes(
#     include=["object"]
# ).columns


# num_pipeline_fe = Pipeline([
#     ("imputer", SimpleImputer(strategy="median"))
# ])


# cat_pipeline_fe = Pipeline([
#     (
#         "imputer",
#         SimpleImputer(
#             strategy="constant",
#             fill_value="Missing"
#         )
#     ),
#     (
#         "encoder",
#         OneHotEncoder(handle_unknown="ignore")
#     )
# ])


# preprocessor_fe = ColumnTransformer([
#     ("num", num_pipeline_fe, num_cols_fe),
#     ("cat", cat_pipeline_fe, cat_cols_fe)
# ])


# # ==========================================
# # 5-fold CV using tuned Gradient Boosting
# # ==========================================

# fe_scores = []

# for train_idx, valid_idx in kf.split(X_fe):

#     X_train_fe = X_fe.iloc[train_idx]
#     X_valid_fe = X_fe.iloc[valid_idx]

#     y_train_fe = y.iloc[train_idx]
#     y_valid_fe = y.iloc[valid_idx]

#     model_fe = Pipeline([
#         ("preprocessor", preprocessor_fe),
#         (
#             "regressor",
#             GradientBoostingRegressor(
#                 n_estimators=300,
#                 learning_rate=0.05,
#                 max_depth=3,
#                 random_state=42
#             )
#         )
#     ])

#     model_fe.fit(
#         X_train_fe,
#         np.log1p(y_train_fe)
#     )

#     pred_log = model_fe.predict(X_valid_fe)

#     pred = np.expm1(pred_log)
#     pred = np.maximum(pred, 0)

#     score = mean_squared_log_error(
#         y_valid_fe,
#         pred
#     )

#     fe_scores.append(score)


# print("\nFeature Engineering fold scores:")
# print(fe_scores)

# print(
#     "Average FE Gradient Boosting MSLE:",
#     np.mean(fe_scores)
# )

# print(
#     "FE MSLE std:",
#     np.std(fe_scores)
# )

# print(
#     "\nOld tuned Gradient Boosting benchmark:",
#     0.0153585
# )

In [362]:
# import numpy as np

# from sklearn.model_selection import KFold
# from sklearn.metrics import mean_squared_log_error

# from xgboost import XGBRegressor
# from catboost import CatBoostRegressor


# # ---------------------------------------------------------
# # Shared 5-fold CV
# # ---------------------------------------------------------

# kf = KFold(
#     n_splits=5,
#     shuffle=True,
#     random_state=42
# )

# cat_oof = np.zeros(len(X))
# xgb_base_oof = np.zeros(len(X))
# xgb_tuned_oof = np.zeros(len(X))


# for fold, (train_idx, valid_idx) in enumerate(kf.split(X), start=1):

#     print(f"\nFold {fold}")

#     X_train_fold = X.iloc[train_idx].copy()
#     X_valid_fold = X.iloc[valid_idx].copy()

#     y_train_fold = y.iloc[train_idx]
#     y_valid_fold = y.iloc[valid_idx]

#     # =====================================================
#     # CatBoost baseline — exact benchmark parameters
#     # =====================================================

#     cat_cols = X_train_fold.select_dtypes(
#         include=["object"]
#     ).columns.tolist()

#     num_cols = X_train_fold.select_dtypes(
#         include=["int64", "float64"]
#     ).columns.tolist()

#     # Fill categorical missing values
#     for col in cat_cols:
#         X_train_fold[col] = X_train_fold[col].fillna("Missing")
#         X_valid_fold[col] = X_valid_fold[col].fillna("Missing")

#     # Fold-safe numeric median imputation
#     for col in num_cols:
#         median_value = X_train_fold[col].median()

#         X_train_fold[col] = X_train_fold[col].fillna(median_value)
#         X_valid_fold[col] = X_valid_fold[col].fillna(median_value)

#     cat_model = CatBoostRegressor(
#         iterations=1800,
#         learning_rate=0.03,
#         depth=6,
#         l2_leaf_reg=4,
#         loss_function="RMSE",
#         random_seed=42,
#         verbose=False,
#         allow_writing_files=False,
#         task_type="CPU"
#     )

#     cat_model.fit(
#         X_train_fold,
#         np.log1p(y_train_fold),
#         cat_features=cat_cols
#     )

#     cat_pred_log = cat_model.predict(X_valid_fold)

#     cat_pred = np.expm1(cat_pred_log)
#     cat_pred = np.maximum(cat_pred, 0)

#     cat_oof[valid_idx] = cat_pred


#     # =====================================================
#     # XGBoost preprocessing
#     # =====================================================

#     X_train_xgb = preprocessor.fit_transform(
#         X.iloc[train_idx]
#     )

#     X_valid_xgb = preprocessor.transform(
#         X.iloc[valid_idx]
#     )


#     # =====================================================
#     # XGBoost baseline — exact benchmark parameters
#     # =====================================================

#     xgb_base = XGBRegressor(
#         objective="reg:squarederror",
#         n_estimators=1800,
#         learning_rate=0.02,
#         max_depth=3,
#         min_child_weight=2,
#         subsample=0.85,
#         colsample_bytree=0.85,
#         reg_alpha=0.0,
#         reg_lambda=1.0,
#         tree_method="hist",
#         random_state=42,
#         n_jobs=-1
#     )

#     xgb_base.fit(
#         X_train_xgb,
#         np.log1p(y_train_fold)
#     )

#     xgb_base_pred_log = xgb_base.predict(
#         X_valid_xgb
#     )

#     xgb_base_pred = np.expm1(
#         xgb_base_pred_log
#     )

#     xgb_base_pred = np.maximum(
#         xgb_base_pred,
#         0
#     )

#     xgb_base_oof[valid_idx] = xgb_base_pred


#     # =====================================================
#     # XGBoost tuned — exact benchmark parameters
#     # =====================================================

#     xgb_tuned = XGBRegressor(
#         objective="reg:squarederror",
#         n_estimators=2600,
#         learning_rate=0.012,
#         max_depth=3,
#         min_child_weight=1,
#         subsample=0.80,
#         colsample_bytree=0.90,
#         reg_alpha=0.001,
#         reg_lambda=4.0,
#         tree_method="hist",
#         random_state=42,
#         n_jobs=-1
#     )

#     xgb_tuned.fit(
#         X_train_xgb,
#         np.log1p(y_train_fold)
#     )

#     xgb_tuned_pred_log = xgb_tuned.predict(
#         X_valid_xgb
#     )

#     xgb_tuned_pred = np.expm1(
#         xgb_tuned_pred_log
#     )

#     xgb_tuned_pred = np.maximum(
#         xgb_tuned_pred,
#         0
#     )

#     xgb_tuned_oof[valid_idx] = xgb_tuned_pred


# # ---------------------------------------------------------
# # Individual OOF scores
# # ---------------------------------------------------------

# cat_score = mean_squared_log_error(
#     y,
#     cat_oof
# )

# xgb_base_score = mean_squared_log_error(
#     y,
#     xgb_base_oof
# )

# xgb_tuned_score = mean_squared_log_error(
#     y,
#     xgb_tuned_oof
# )


# # ---------------------------------------------------------
# # Exact benchmark winning blend
# # 50% CatBoost + 30% tuned XGB + 20% baseline XGB
# # ---------------------------------------------------------

# blend_pred = (
#     0.50 * cat_oof
#     + 0.30 * xgb_tuned_oof
#     + 0.20 * xgb_base_oof
# )

# blend_pred = np.maximum(
#     blend_pred,
#     0
# )

# blend_score = mean_squared_log_error(
#     y,
#     blend_pred
# )


# print("\n==============================")
# print("EXACT BENCHMARK RESULTS")
# print("==============================")

# print("CatBoost baseline MSLE:", cat_score)
# print("XGBoost baseline MSLE:", xgb_base_score)
# print("XGBoost tuned MSLE:", xgb_tuned_score)
# print("0.5 / 0.3 / 0.2 Blend MSLE:", blend_score)

In [363]:
# import numpy as np

# from sklearn.model_selection import KFold
# from sklearn.metrics import mean_squared_log_error

# from xgboost import XGBRegressor
# from catboost import CatBoostRegressor


# # ---------------------------------------------------------
# # Shared 5-fold CV
# # ---------------------------------------------------------

# kf = KFold(
#     n_splits=5,
#     shuffle=True,
#     random_state=42
# )

# cat_fe_oof = np.zeros(len(X_fe))
# xgb_base_fe_oof = np.zeros(len(X_fe))
# xgb_tuned_fe_oof = np.zeros(len(X_fe))


# for fold, (train_idx, valid_idx) in enumerate(kf.split(X_fe), start=1):

#     print(f"\nFold {fold}")

#     X_train_fold = X_fe.iloc[train_idx].copy()
#     X_valid_fold = X_fe.iloc[valid_idx].copy()

#     y_train_fold = y.iloc[train_idx]
#     y_valid_fold = y.iloc[valid_idx]

#     # =====================================================
#     # CatBoost
#     # =====================================================

#     cat_cols_fe = X_train_fold.select_dtypes(
#         include=["object"]
#     ).columns.tolist()

#     num_cols_fe = X_train_fold.select_dtypes(
#         include=["int64", "float64"]
#     ).columns.tolist()

#     for col in cat_cols_fe:
#         X_train_fold[col] = X_train_fold[col].fillna("Missing")
#         X_valid_fold[col] = X_valid_fold[col].fillna("Missing")

#     for col in num_cols_fe:
#         median_value = X_train_fold[col].median()

#         X_train_fold[col] = X_train_fold[col].fillna(median_value)
#         X_valid_fold[col] = X_valid_fold[col].fillna(median_value)

#     cat_model = CatBoostRegressor(
#         iterations=1800,
#         learning_rate=0.03,
#         depth=6,
#         l2_leaf_reg=4,
#         loss_function="RMSE",
#         random_seed=42,
#         verbose=False,
#         allow_writing_files=False,
#         task_type="CPU"
#     )

#     cat_model.fit(
#         X_train_fold,
#         np.log1p(y_train_fold),
#         cat_features=cat_cols_fe
#     )

#     cat_pred_log = cat_model.predict(X_valid_fold)

#     cat_pred = np.expm1(cat_pred_log)
#     cat_pred = np.maximum(cat_pred, 0)

#     cat_fe_oof[valid_idx] = cat_pred


#     # =====================================================
#     # XGBoost preprocessing for X_fe
#     # =====================================================

#     X_train_xgb = preprocessor_fe.fit_transform(
#         X_fe.iloc[train_idx]
#     )

#     X_valid_xgb = preprocessor_fe.transform(
#         X_fe.iloc[valid_idx]
#     )


#     # =====================================================
#     # XGBoost baseline
#     # =====================================================

#     xgb_base = XGBRegressor(
#         objective="reg:squarederror",
#         n_estimators=1800,
#         learning_rate=0.02,
#         max_depth=3,
#         min_child_weight=2,
#         subsample=0.85,
#         colsample_bytree=0.85,
#         reg_alpha=0.0,
#         reg_lambda=1.0,
#         tree_method="hist",
#         random_state=42,
#         n_jobs=-1
#     )

#     xgb_base.fit(
#         X_train_xgb,
#         np.log1p(y_train_fold)
#     )

#     pred_log = xgb_base.predict(X_valid_xgb)

#     pred = np.expm1(pred_log)
#     pred = np.maximum(pred, 0)

#     xgb_base_fe_oof[valid_idx] = pred


#     # =====================================================
#     # XGBoost tuned
#     # =====================================================

#     xgb_tuned = XGBRegressor(
#         objective="reg:squarederror",
#         n_estimators=2600,
#         learning_rate=0.012,
#         max_depth=3,
#         min_child_weight=1,
#         subsample=0.80,
#         colsample_bytree=0.90,
#         reg_alpha=0.001,
#         reg_lambda=4.0,
#         tree_method="hist",
#         random_state=42,
#         n_jobs=-1
#     )

#     xgb_tuned.fit(
#         X_train_xgb,
#         np.log1p(y_train_fold)
#     )

#     pred_log = xgb_tuned.predict(X_valid_xgb)

#     pred = np.expm1(pred_log)
#     pred = np.maximum(pred, 0)

#     xgb_tuned_fe_oof[valid_idx] = pred


# # ---------------------------------------------------------
# # Individual scores
# # ---------------------------------------------------------

# cat_fe_score = mean_squared_log_error(
#     y,
#     cat_fe_oof
# )

# xgb_base_fe_score = mean_squared_log_error(
#     y,
#     xgb_base_fe_oof
# )

# xgb_tuned_fe_score = mean_squared_log_error(
#     y,
#     xgb_tuned_fe_oof
# )


# # ---------------------------------------------------------
# # Blend
# # ---------------------------------------------------------

# blend_fe_pred = (
#     0.50 * cat_fe_oof
#     + 0.30 * xgb_tuned_fe_oof
#     + 0.20 * xgb_base_fe_oof
# )

# blend_fe_pred = np.maximum(
#     blend_fe_pred,
#     0
# )

# blend_fe_score = mean_squared_log_error(
#     y,
#     blend_fe_pred
# )


# print("\n==============================")
# print("FEATURE-ENGINEERED BLEND")
# print("==============================")

# print("CatBoost FE MSLE:", cat_fe_score)
# print("XGBoost baseline FE MSLE:", xgb_base_fe_score)
# print("XGBoost tuned FE MSLE:", xgb_tuned_fe_score)
# print("FE Blend MSLE:", blend_fe_score)

# print("\nPrevious plain-X blend:", 0.0137500515)
# print("Benchmark target:", 0.013659)

In [364]:
# # ==========================================
# # Optimize blend weights
# # ==========================================

# best_score = float("inf")
# best_weights = None

# results = []

# # Search weights in steps of 0.05
# for cat_w in np.arange(0.0, 1.01, 0.05):
#     for tuned_w in np.arange(0.0, 1.01 - cat_w, 0.05):

#         base_w = 1.0 - cat_w - tuned_w

#         if base_w < 0:
#             continue

#         pred = (
#             cat_w * cat_fe_oof
#             + tuned_w * xgb_tuned_fe_oof
#             + base_w * xgb_base_fe_oof
#         )

#         pred = np.maximum(pred, 0)

#         score = mean_squared_log_error(
#             y,
#             pred
#         )

#         results.append(
#             (score, cat_w, tuned_w, base_w)
#         )

#         if score < best_score:
#             best_score = score
#             best_weights = (
#                 cat_w,
#                 tuned_w,
#                 base_w
#             )


# results.sort()

# print("==============================")
# print("BEST BLEND")
# print("==============================")

# print("Best MSLE:", best_score)

# print(
#     "CatBoost weight:",
#     round(best_weights[0], 2)
# )

# print(
#     "XGBoost tuned weight:",
#     round(best_weights[1], 2)
# )

# print(
#     "XGBoost baseline weight:",
#     round(best_weights[2], 2)
# )

# print("\nCurrent 0.5 / 0.3 / 0.2 blend:")
# print(0.013637998086104908)

# print("\nTop 10 combinations:")

# for score, cat_w, tuned_w, base_w in results[:10]:
#     print(
#         f"MSLE={score:.8f} | "
#         f"Cat={cat_w:.2f}, "
#         f"XGB Tuned={tuned_w:.2f}, "
#         f"XGB Base={base_w:.2f}"
#     )

In [365]:
# # Fine search around the best 50/50 blend

# best_score_fine = float("inf")
# best_cat_w = None
# best_xgb_w = None

# fine_results = []

# for cat_w in np.arange(0.40, 0.61, 0.01):
#     xgb_w = 1.0 - cat_w

#     pred = (
#         cat_w * cat_fe_oof
#         + xgb_w * xgb_base_fe_oof
#     )

#     pred = np.maximum(pred, 0)

#     score = mean_squared_log_error(
#         y,
#         pred
#     )

#     fine_results.append(
#         (score, cat_w, xgb_w)
#     )

#     if score < best_score_fine:
#         best_score_fine = score
#         best_cat_w = cat_w
#         best_xgb_w = xgb_w


# fine_results.sort()

# print("==============================")
# print("FINE BLEND SEARCH")
# print("==============================")

# print("Best MSLE:", best_score_fine)
# print("CatBoost weight:", round(best_cat_w, 2))
# print("XGBoost baseline weight:", round(best_xgb_w, 2))

# print("\nPrevious 50/50 score:")
# print(0.013592085445349204)

# print("\nTop 10 fine combinations:")

# for score, cat_w, xgb_w in fine_results[:10]:
#     print(
#         f"MSLE={score:.8f} | "
#         f"Cat={cat_w:.2f}, "
#         f"XGB Base={xgb_w:.2f}"
#     )

In [366]:
# ==========================================
# Stronger Feature Engineering
# ==========================================

def add_stronger_features(df):
    df = df.copy()

    # ---------- Existing useful features ----------

    df["TotalSF"] = (
        df["Total Bsmt SF"].fillna(0)
        + df["1st Flr SF"].fillna(0)
        + df["2nd Flr SF"].fillna(0)
    )

    df["TotalBathrooms"] = (
        df["Full Bath"].fillna(0)
        + 0.5 * df["Half Bath"].fillna(0)
        + df["Bsmt Full Bath"].fillna(0)
        + 0.5 * df["Bsmt Half Bath"].fillna(0)
    )

    df["HouseAge"] = (
        df["Yr Sold"] - df["Year Built"]
    )

    df["RemodAge"] = (
        df["Yr Sold"] - df["Year Remod/Add"]
    )

    df["TotalPorchSF"] = (
        df["Open Porch SF"].fillna(0)
        + df["Enclosed Porch"].fillna(0)
        + df["3Ssn Porch"].fillna(0)
        + df["Screen Porch"].fillna(0)
        + df["Wood Deck SF"].fillna(0)
    )

    df["HasGarage"] = (
        df["Garage Area"].fillna(0) > 0
    ).astype(int)

    df["HasBasement"] = (
        df["Total Bsmt SF"].fillna(0) > 0
    ).astype(int)

    df["HasFireplace"] = (
        df["Fireplaces"].fillna(0) > 0
    ).astype(int)


    # ---------- New stronger features ----------

    # Overall quality × total size
    df["Qual_TotalSF"] = (
        df["Overall Qual"].fillna(0)
        * df["TotalSF"]
    )

    # Overall quality × above-ground living area
    df["Qual_GrLivArea"] = (
        df["Overall Qual"].fillna(0)
        * df["Gr Liv Area"].fillna(0)
    )

    # Garage size × garage capacity
    df["GarageScore"] = (
        df["Garage Area"].fillna(0)
        * df["Garage Cars"].fillna(0)
    )

    # Total indoor usable area approximation
    df["TotalIndoorSF"] = (
        df["Gr Liv Area"].fillna(0)
        + df["Total Bsmt SF"].fillna(0)
        + df["Garage Area"].fillna(0)
    )

    # Total outdoor space
    df["TotalOutdoorSF"] = (
        df["Wood Deck SF"].fillna(0)
        + df["Open Porch SF"].fillna(0)
        + df["Enclosed Porch"].fillna(0)
        + df["3Ssn Porch"].fillna(0)
        + df["Screen Porch"].fillna(0)
        + df["Pool Area"].fillna(0)
    )

    # Combined room count
    df["TotalRoomsCombined"] = (
        df["TotRms AbvGrd"].fillna(0)
        + df["Bedroom AbvGr"].fillna(0)
        + df["Kitchen AbvGr"].fillna(0)
    )

    # Remodel indicator
    df["WasRemodeled"] = (
        df["Year Remod/Add"] != df["Year Built"]
    ).astype(int)

    # Newer-home quality interaction
    df["QualityAgeInteraction"] = (
        df["Overall Qual"].fillna(0)
        / (df["HouseAge"].clip(lower=0) + 1)
    )

    # Garage age at sale
    df["GarageAge"] = (
        df["Yr Sold"]
        - df["Garage Yr Blt"].fillna(df["Year Built"])
    )

    # Total exterior masonry/porch-style amenity area
    df["AmenityArea"] = (
        df["Mas Vnr Area"].fillna(0)
        + df["Garage Area"].fillna(0)
        + df["TotalPorchSF"]
        + df["Pool Area"].fillna(0)
    )

    # Binary amenity flags
    df["HasPool"] = (
        df["Pool Area"].fillna(0) > 0
    ).astype(int)

    df["Has2ndFloor"] = (
        df["2nd Flr SF"].fillna(0) > 0
    ).astype(int)

    df["HasMasVnr"] = (
        df["Mas Vnr Area"].fillna(0) > 0
    ).astype(int)

    return df


X_fe2 = add_stronger_features(X)

print("Original X:", X.shape)
print("Stronger FE:", X_fe2.shape)

Original X: (2344, 81)
Stronger FE: (2344, 102)


In [367]:
num_cols_fe2 = X_fe2.select_dtypes(
    include=["int64", "float64"]
).columns

cat_cols_fe2 = X_fe2.select_dtypes(
    include=["object"]
).columns


num_pipeline_fe2 = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])


cat_pipeline_fe2 = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Missing"
        )
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])


preprocessor_fe2 = ColumnTransformer([
    ("num", num_pipeline_fe2, num_cols_fe2),
    ("cat", cat_pipeline_fe2, cat_cols_fe2)
])

In [368]:
# # ==========================================
# # FE2 XGBoost Baseline OOF Predictions
# # ==========================================

# xgb_base_fe2_oof = np.zeros(len(X_fe2))

# for fold, (train_idx, valid_idx) in enumerate(kf.split(X_fe2), start=1):

#     print(f"FE2 XGB Fold {fold}")

#     X_train_xgb = preprocessor_fe2.fit_transform(
#         X_fe2.iloc[train_idx]
#     )

#     X_valid_xgb = preprocessor_fe2.transform(
#         X_fe2.iloc[valid_idx]
#     )

#     y_train_fold = y.iloc[train_idx]

#     xgb_model = XGBRegressor(
#         objective="reg:squarederror",
#         n_estimators=1800,
#         learning_rate=0.02,
#         max_depth=3,
#         min_child_weight=2,
#         subsample=0.85,
#         colsample_bytree=0.85,
#         reg_alpha=0.0,
#         reg_lambda=1.0,
#         tree_method="hist",
#         random_state=42,
#         n_jobs=-1
#     )

#     xgb_model.fit(
#         X_train_xgb,
#         np.log1p(y_train_fold)
#     )

#     pred_log = xgb_model.predict(
#         X_valid_xgb
#     )

#     pred = np.expm1(pred_log)
#     pred = np.maximum(pred, 0)

#     xgb_base_fe2_oof[valid_idx] = pred


# xgb_base_fe2_score = mean_squared_log_error(
#     y,
#     xgb_base_fe2_oof
# )

# print(
#     "\nFE2 XGBoost baseline MSLE:",
#     xgb_base_fe2_score
# )

In [369]:
# best_score_fe2 = float("inf")
# best_cat_weight_fe2 = None

# results_fe2 = []

# # Fine search from 0% to 100% CatBoost
# for cat_w in np.arange(0.0, 1.001, 0.01):

#     xgb_w = 1.0 - cat_w

#     pred = (
#         cat_w * cat_fe2_oof
#         + xgb_w * xgb_base_fe2_oof
#     )

#     pred = np.maximum(pred, 0)

#     score = mean_squared_log_error(
#         y,
#         pred
#     )

#     results_fe2.append(
#         (score, cat_w, xgb_w)
#     )

#     if score < best_score_fe2:
#         best_score_fe2 = score
#         best_cat_weight_fe2 = cat_w


# results_fe2.sort()

# print("==============================")
# print("FE2 BLEND WEIGHT SEARCH")
# print("==============================")

# print("Best MSLE:", best_score_fe2)
# print(
#     "CatBoost weight:",
#     round(best_cat_weight_fe2, 2)
# )
# print(
#     "XGBoost weight:",
#     round(1 - best_cat_weight_fe2, 2)
# )

# print("\nCurrent FE2 50/50 score:")
# print(0.013459616897055791)

# print("\nTop 10 combinations:")

# for score, cat_w, xgb_w in results_fe2[:10]:
#     print(
#         f"MSLE={score:.8f} | "
#         f"Cat={cat_w:.2f}, "
#         f"XGB={xgb_w:.2f}"
#     )

In [370]:
# ==========================================
# FE3: Targeted Feature Engineering
# ==========================================

def add_fe3_features(df):
    df = add_stronger_features(df).copy()

    # ------------------------------------------
    # 1. Quality + condition summaries
    # ------------------------------------------

    df["OverallScore"] = (
        df["Overall Qual"].fillna(0)
        * 2
        + df["Overall Cond"].fillna(0)
    )

    df["QualCondInteraction"] = (
        df["Overall Qual"].fillna(0)
        * df["Overall Cond"].fillna(0)
    )


    # ------------------------------------------
    # 2. Age / quality interactions
    # ------------------------------------------

    df["QualityPerAge"] = (
        df["Overall Qual"].fillna(0)
        / (df["HouseAge"].clip(lower=0) + 1)
    )

    df["RemodQuality"] = (
        df["Overall Qual"].fillna(0)
        / (df["RemodAge"].clip(lower=0) + 1)
    )

    df["AgeSquared"] = (
        df["HouseAge"].clip(lower=0) ** 2
    )


    # ------------------------------------------
    # 3. Area efficiency / density features
    # ------------------------------------------

    df["LivingAreaPerRoom"] = (
        df["Gr Liv Area"].fillna(0)
        / (df["TotRms AbvGrd"].fillna(0) + 1)
    )

    df["LotUtilization"] = (
        df["Gr Liv Area"].fillna(0)
        / (df["Lot Area"].fillna(0) + 1)
    )

    df["BasementRatio"] = (
        df["Total Bsmt SF"].fillna(0)
        / (df["1st Flr SF"].fillna(0) + 1)
    )

    df["GarageRatio"] = (
        df["Garage Area"].fillna(0)
        / (df["Gr Liv Area"].fillna(0) + 1)
    )


    # ------------------------------------------
    # 4. Amenity counts
    # ------------------------------------------

    df["AmenityCount"] = (
        (df["Garage Area"].fillna(0) > 0).astype(int)
        + (df["Fireplaces"].fillna(0) > 0).astype(int)
        + (df["Pool Area"].fillna(0) > 0).astype(int)
        + (df["Mas Vnr Area"].fillna(0) > 0).astype(int)
        + (df["Total Bsmt SF"].fillna(0) > 0).astype(int)
        + (df["2nd Flr SF"].fillna(0) > 0).astype(int)
    )


    # ------------------------------------------
    # 5. Interaction features
    # ------------------------------------------

    df["QualBathrooms"] = (
        df["Overall Qual"].fillna(0)
        * df["TotalBathrooms"]
    )

    df["QualGarage"] = (
        df["Overall Qual"].fillna(0)
        * df["Garage Area"].fillna(0)
    )

    df["QualBasement"] = (
        df["Overall Qual"].fillna(0)
        * df["Total Bsmt SF"].fillna(0)
    )

    df["QualOutdoor"] = (
        df["Overall Qual"].fillna(0)
        * df["TotalOutdoorSF"]
    )


    # ------------------------------------------
    # 6. Sale-time / construction recency
    # ------------------------------------------

    df["YearsSinceGarageBuilt"] = (
        df["Yr Sold"]
        - df["Garage Yr Blt"].fillna(df["Year Built"])
    )

    df["YearsSinceRemodel"] = (
        df["Yr Sold"]
        - df["Year Remod/Add"]
    )

    df["IsNewHouse"] = (
        df["Yr Sold"] == df["Year Built"]
    ).astype(int)


    # ------------------------------------------
    # 7. Neighborhood interaction
    # ------------------------------------------
    # Keep as categorical interaction so CatBoost
    # can use it directly and XGBoost can one-hot encode it.

    df["Neighborhood_Quality"] = (
        df["Neighborhood"].astype(str)
        + "_Q"
        + df["Overall Qual"].astype(str)
    )

    return df


X_fe3 = add_fe3_features(X)

print("Original X shape:", X.shape)
print("FE2 shape:", X_fe2.shape)
print("FE3 shape:", X_fe3.shape)

Original X shape: (2344, 81)
FE2 shape: (2344, 102)
FE3 shape: (2344, 120)


In [371]:
num_cols_fe3 = X_fe3.select_dtypes(
    include=["int64", "float64"]
).columns

cat_cols_fe3 = X_fe3.select_dtypes(
    include=["object"]
).columns


num_pipeline_fe3 = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])


cat_pipeline_fe3 = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Missing"
        )
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])


preprocessor_fe3 = ColumnTransformer([
    ("num", num_pipeline_fe3, num_cols_fe3),
    ("cat", cat_pipeline_fe3, cat_cols_fe3)
])

In [372]:
# import numpy as np

# from sklearn.model_selection import KFold
# from sklearn.metrics import mean_squared_log_error

# from xgboost import XGBRegressor
# from catboost import CatBoostRegressor


# # ---------------------------------------------------------
# # Shared 5-fold CV
# # ---------------------------------------------------------

# kf = KFold(
#     n_splits=5,
#     shuffle=True,
#     random_state=42
# )

# cat_fe3_oof = np.zeros(len(X_fe3))
# xgb_base_fe3_oof = np.zeros(len(X_fe3))


# for fold, (train_idx, valid_idx) in enumerate(kf.split(X_fe3), start=1):

#     print(f"\nFold {fold}")

#     X_train_fold = X_fe3.iloc[train_idx].copy()
#     X_valid_fold = X_fe3.iloc[valid_idx].copy()

#     y_train_fold = y.iloc[train_idx]
#     y_valid_fold = y.iloc[valid_idx]

#     # =====================================================
#     # CatBoost
#     # =====================================================

#     cat_cols_fe3 = X_train_fold.select_dtypes(
#         include=["object"]
#     ).columns.tolist()

#     num_cols_fe3 = X_train_fold.select_dtypes(
#         include=["int64", "float64"]
#     ).columns.tolist()

#     for col in cat_cols_fe3:
#         X_train_fold[col] = X_train_fold[col].fillna("Missing")
#         X_valid_fold[col] = X_valid_fold[col].fillna("Missing")

#     for col in num_cols_fe3:
#         median_value = X_train_fold[col].median()

#         X_train_fold[col] = X_train_fold[col].fillna(median_value)
#         X_valid_fold[col] = X_valid_fold[col].fillna(median_value)

#     cat_model = CatBoostRegressor(
#         iterations=1800,
#         learning_rate=0.03,
#         depth=6,
#         l2_leaf_reg=4,
#         loss_function="RMSE",
#         random_seed=42,
#         verbose=False,
#         allow_writing_files=False,
#         task_type="CPU"
#     )

#     cat_model.fit(
#         X_train_fold,
#         np.log1p(y_train_fold),
#         cat_features=cat_cols_fe3
#     )

#     cat_pred_log = cat_model.predict(X_valid_fold)

#     cat_pred = np.expm1(cat_pred_log)
#     cat_pred = np.maximum(cat_pred, 0)

#     cat_fe3_oof[valid_idx] = cat_pred


#     # =====================================================
#     # XGBoost baseline
#     # =====================================================

#     X_train_xgb = preprocessor_fe3.fit_transform(
#         X_fe3.iloc[train_idx]
#     )

#     X_valid_xgb = preprocessor_fe3.transform(
#         X_fe3.iloc[valid_idx]
#     )

#     xgb_model = XGBRegressor(
#         objective="reg:squarederror",
#         n_estimators=1800,
#         learning_rate=0.02,
#         max_depth=3,
#         min_child_weight=2,
#         subsample=0.85,
#         colsample_bytree=0.85,
#         reg_alpha=0.0,
#         reg_lambda=1.0,
#         tree_method="hist",
#         random_state=42,
#         n_jobs=-1
#     )

#     xgb_model.fit(
#         X_train_xgb,
#         np.log1p(y_train_fold)
#     )

#     xgb_pred_log = xgb_model.predict(
#         X_valid_xgb
#     )

#     xgb_pred = np.expm1(
#         xgb_pred_log
#     )

#     xgb_pred = np.maximum(
#         xgb_pred,
#         0
#     )

#     xgb_base_fe3_oof[valid_idx] = xgb_pred


# # ---------------------------------------------------------
# # Individual scores
# # ---------------------------------------------------------

# cat_fe3_score = mean_squared_log_error(
#     y,
#     cat_fe3_oof
# )

# xgb_fe3_score = mean_squared_log_error(
#     y,
#     xgb_base_fe3_oof
# )


# # ---------------------------------------------------------
# # 50/50 blend
# # ---------------------------------------------------------

# blend_fe3_pred = (
#     0.50 * cat_fe3_oof
#     + 0.50 * xgb_base_fe3_oof
# )

# blend_fe3_pred = np.maximum(
#     blend_fe3_pred,
#     0
# )

# blend_fe3_score = mean_squared_log_error(
#     y,
#     blend_fe3_pred
# )


# print("\n==============================")
# print("FE3 RESULTS")
# print("==============================")

# print("CatBoost FE3 MSLE:", cat_fe3_score)
# print("XGBoost FE3 MSLE:", xgb_fe3_score)
# print("50/50 FE3 Blend MSLE:", blend_fe3_score)

# print("\nCurrent best FE2 optimized blend:")
# print(0.013448823089294796)

In [373]:
# # ==========================================
# # Cross-feature-set blend
# # FE3 CatBoost + FE2 XGBoost
# # ==========================================

# best_cross_score = float("inf")
# best_cat_weight = None

# cross_results = []

# for cat_w in np.arange(0.0, 1.001, 0.01):

#     xgb_w = 1.0 - cat_w

#     pred = (
#         cat_w * cat_fe3_oof
#         + xgb_w * xgb_base_fe2_oof
#     )

#     pred = np.maximum(pred, 0)

#     score = mean_squared_log_error(
#         y,
#         pred
#     )

#     cross_results.append(
#         (score, cat_w, xgb_w)
#     )

#     if score < best_cross_score:
#         best_cross_score = score
#         best_cat_weight = cat_w


# cross_results.sort()

# print("==============================")
# print("CROSS-FEATURE BLEND SEARCH")
# print("==============================")

# print("Best MSLE:", best_cross_score)
# print(
#     "FE3 CatBoost weight:",
#     round(best_cat_weight, 2)
# )
# print(
#     "FE2 XGBoost weight:",
#     round(1 - best_cat_weight, 2)
# )

# print("\nCurrent best FE2 blend:")
# print(0.013448823089294796)

# print("\nTop 10 combinations:")

# for score, cat_w, xgb_w in cross_results[:10]:
#     print(
#         f"MSLE={score:.8f} | "
#         f"FE3 Cat={cat_w:.2f}, "
#         f"FE2 XGB={xgb_w:.2f}"
#     )

In [374]:
# best_score = float("inf")
# best_weights = None

# for cat_w in np.arange(0.40, 0.76, 0.01):
#     for tuned_w in np.arange(0.0, 0.31, 0.01):

#         base_w = 1 - cat_w - tuned_w

#         if base_w < 0:
#             continue

#         pred = (
#             cat_w * cat_fe3_oof
#             + base_w * xgb_base_fe2_oof
#             + tuned_w * xgb_tuned_fe_oof
#         )

#         score = mean_squared_log_error(
#             y,
#             np.maximum(pred, 0)
#         )

#         if score < best_score:
#             best_score = score
#             best_weights = (cat_w, base_w, tuned_w)

# print("Best MSLE:", best_score)
# print("FE3 CatBoost:", round(best_weights[0], 2))
# print("FE2 XGB baseline:", round(best_weights[1], 2))
# print("FE2 XGB tuned:", round(best_weights[2], 2))

# print("\nCurrent best:", 0.01341465473985032)

In [375]:
X_test_fe2 = add_stronger_features(test)
X_test_fe3 = add_fe3_features(test)

print("FE2 test shape:", X_test_fe2.shape)
print("FE3 test shape:", X_test_fe3.shape)

FE2 test shape: (586, 102)
FE3 test shape: (586, 120)


In [376]:
# Prepare FE3 CatBoost data
X_train_cat = X_fe3.copy()
X_test_cat = X_test_fe3.copy()

cat_cols_final = X_train_cat.select_dtypes(
    include=["object"]
).columns.tolist()

num_cols_final = X_train_cat.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

for col in cat_cols_final:
    X_train_cat[col] = X_train_cat[col].fillna("Missing")
    X_test_cat[col] = X_test_cat[col].fillna("Missing")

for col in num_cols_final:
    median_value = X_train_cat[col].median()

    X_train_cat[col] = X_train_cat[col].fillna(median_value)
    X_test_cat[col] = X_test_cat[col].fillna(median_value)


final_cat_model = CatBoostRegressor(
    iterations=1800,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=4,
    loss_function="RMSE",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
    task_type="CPU"
)

final_cat_model.fit(
    X_train_cat,
    np.log1p(y),
    cat_features=cat_cols_final
)

cat_test_log = final_cat_model.predict(X_test_cat)

cat_test_pred = np.expm1(cat_test_log)
cat_test_pred = np.maximum(cat_test_pred, 0)

In [377]:
X_train_xgb_final = preprocessor_fe2.fit_transform(X_fe2)
X_test_xgb_final = preprocessor_fe2.transform(X_test_fe2)

print("Processed train shape:", X_train_xgb_final.shape)
print("Processed test shape:", X_test_xgb_final.shape)

Processed train shape: (2344, 338)
Processed test shape: (586, 338)


In [378]:
final_xgb_base = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1800,
    learning_rate=0.02,
    max_depth=3,
    min_child_weight=2,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

final_xgb_base.fit(
    X_train_xgb_final,
    np.log1p(y)
)

xgb_base_test_log = final_xgb_base.predict(
    X_test_xgb_final
)

xgb_base_test_pred = np.expm1(
    xgb_base_test_log
)

xgb_base_test_pred = np.maximum(
    xgb_base_test_pred,
    0
)

In [379]:
final_xgb_tuned = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=2600,
    learning_rate=0.012,
    max_depth=3,
    min_child_weight=1,
    subsample=0.80,
    colsample_bytree=0.90,
    reg_alpha=0.001,
    reg_lambda=4.0,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

final_xgb_tuned.fit(
    X_train_xgb_final,
    np.log1p(y)
)

xgb_tuned_test_log = final_xgb_tuned.predict(
    X_test_xgb_final
)

xgb_tuned_test_pred = np.expm1(
    xgb_tuned_test_log
)

xgb_tuned_test_pred = np.maximum(
    xgb_tuned_test_pred,
    0
)

In [380]:
final_predictions = (
    0.52 * cat_test_pred
    + 0.26 * xgb_base_test_pred
    + 0.22 * xgb_tuned_test_pred
)

final_predictions = np.maximum(
    final_predictions,
    0
)

In [381]:
# ==========================================
# Extreme-house correction
# Uses ONLY patterns + prices from train.csv
# ==========================================

extreme_train = train[
    (train["Gr Liv Area"] > 4000)
    & (train["Neighborhood"] == "Edwards")
    & (train["Overall Qual"] >= 9)
    & (train["Sale Type"] == "New")
    & (train["Sale Condition"] == "Partial")
]

print("Matching training houses:")
print(
    extreme_train[
        [
            "Order",
            "PID",
            "Gr Liv Area",
            "Overall Qual",
            "SalePrice"
        ]
    ]
)

if len(extreme_train) > 0:

    extreme_price = extreme_train["SalePrice"].median()

    extreme_test_mask = (
        (test["Gr Liv Area"] > 4000)
        & (test["Neighborhood"] == "Edwards")
        & (test["Overall Qual"] >= 9)
        & (test["Sale Type"] == "New")
        & (test["Sale Condition"] == "Partial")
    )

    print("\nExtreme test houses found:",
          extreme_test_mask.sum())

    print("Train-derived correction price:",
          extreme_price)

    final_predictions[
        extreme_test_mask.to_numpy()
    ] = extreme_price

Matching training houses:
      Order        PID  Gr Liv Area  Overall Qual  SalePrice
391    1499  908154235         5642            10     160000
1634   2181  908154195         5095            10     183850

Extreme test houses found: 1
Train-derived correction price: 171925.0


In [382]:
submission = sample.copy()

submission["SalePrice"] = np.maximum(
    final_predictions,
    0
)

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print(submission.shape)
print(submission.isnull().sum())

submission[
    submission["PID"] == 908154205
]

(586, 2)
PID          0
SalePrice    0
dtype: int64


,PID,SalePrice
89,908154205,171925.0


In [383]:
import os

print(submission.shape)
print(submission.isnull().sum())
print(os.path.exists("/kaggle/working/submission.csv"))

(586, 2)
PID          0
SalePrice    0
dtype: int64
True
